# Data Simulation for Real-Time UPI Fraud Detection

This notebook creates a realistic synthetic UPI transaction dataset for anomaly detection. It follows the project statement: learn each user's normal transaction behavior and flag transactions that deviate from that learned pattern.

The simulated data includes normal transactions and crafted suspicious patterns such as late-night transfers, new beneficiaries, high-value amounts, unusual pincodes, and rapid transaction bursts.

## 1. Imports and Project Paths

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SYNTHETIC_DATA_DIR = DATA_DIR / "synthetic"
RAW_DATA_DIR = DATA_DIR / "raw"

SYNTHETIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 2. Simulation Configuration

In [2]:
@dataclass(frozen=True)
class SimulationConfig:
    """Configuration for synthetic UPI transaction generation."""

    normal_transactions: int = 9700
    fraud_transactions: int = 300
    start_date: str = "2024-01-01"
    normal_period_days: int = 90
    fraud_start_day: int = 90
    fraud_hour_start: int = 1
    fraud_hour_end: int = 5
    random_state: int = RANDOM_STATE

    @property
    def total_transactions(self) -> int:
        return self.normal_transactions + self.fraud_transactions


config = SimulationConfig()
rng = np.random.default_rng(config.random_state)

config

SimulationConfig(normal_transactions=9700, fraud_transactions=300, start_date='2024-01-01', normal_period_days=90, fraud_start_day=90, fraud_hour_start=1, fraud_hour_end=5, random_state=42)

## 3. Helper Functions

In [3]:
def make_account_ids(prefix: str, account_numbers: np.ndarray) -> list[str]:
    """Convert numeric account identifiers into anonymized account IDs."""
    return [f"{prefix}_{account_number:04d}" for account_number in account_numbers]


def make_transaction_ids(prefix: str, count: int) -> list[str]:
    """Create stable transaction IDs for synthetic records."""
    return [f"{prefix}_{index:06d}" for index in range(1, count + 1)]


def generate_random_timestamps(
    start_date: str,
    periods: int,
    count: int,
    frequency: str,
    random_generator: np.random.Generator,
) -> pd.Series:
    """Sample timestamps from a date range."""
    date_range = pd.date_range(start=start_date, periods=periods, freq=frequency)
    sampled_values = random_generator.choice(date_range, size=count, replace=True)
    return pd.Series(pd.to_datetime(sampled_values))


def validate_transactions(transactions: pd.DataFrame) -> None:
    """Validate the generated transaction dataset before saving."""
    required_columns = {
        "transaction_id",
        "timestamp",
        "sender_account_id",
        "receiver_account_id",
        "amount",
        "location_pincode",
        "transaction_type",
        "is_fraud",
    }
    missing_columns = required_columns.difference(transactions.columns)

    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
    if transactions["transaction_id"].duplicated().any():
        raise ValueError("Transaction IDs must be unique.")
    if (transactions["amount"] <= 0).any():
        raise ValueError("Transaction amounts must be positive.")

## 4. Generate Normal Transactions

In [4]:
def generate_normal_transactions(
    config: SimulationConfig,
    random_generator: np.random.Generator,
) -> pd.DataFrame:
    """Generate common UPI transactions that represent normal user behavior."""
    count = config.normal_transactions
    normal_timestamps = generate_random_timestamps(
        start_date=config.start_date,
        periods=config.normal_period_days * 24 * 12,
        count=count,
        frequency="5min",
        random_generator=random_generator,
    )

    sender_accounts = make_account_ids(
        prefix="ACC",
        account_numbers=random_generator.integers(1, 5001, size=count),
    )
    receiver_accounts = make_account_ids(
        prefix="RCV",
        account_numbers=random_generator.integers(1, 8001, size=count),
    )

    return pd.DataFrame(
        {
            "transaction_id": make_transaction_ids("TXN", count),
            "timestamp": normal_timestamps,
            "sender_account_id": sender_accounts,
            "receiver_account_id": receiver_accounts,
            "amount": random_generator.gamma(shape=2.2, scale=850, size=count).round(2),
            "location_pincode": random_generator.choice(
                ["400001", "110001", "560001", "700001", "600001"],
                size=count,
                p=[0.26, 0.22, 0.22, 0.16, 0.14],
            ),
            "transaction_type": random_generator.choice(
                ["P2P", "Merchant"],
                size=count,
                p=[0.60, 0.40],
            ),
            "is_fraud": 0,
        }
    )


normal_transactions = generate_normal_transactions(config, rng)
normal_transactions.head()

,transaction_id,timestamp,sender_account_id,receiver_account_id,amount,location_pincode,transaction_type,is_fraud
0,TXN_000001,2024-01-09 00:45:00,ACC_0174,RCV_2795,843.19,110001,Merchant,0
1,TXN_000002,2024-03-10 15:40:00,ACC_1009,RCV_0519,1274.56,110001,P2P,0
2,TXN_000003,2024-02-28 21:50:00,ACC_1852,RCV_4881,417.11,560001,P2P,0
3,TXN_000004,2024-02-09 11:55:00,ACC_3657,RCV_0923,2023.92,600001,Merchant,0
4,TXN_000005,2024-02-08 23:15:00,ACC_0371,RCV_1736,1323.01,400001,P2P,0


## 5. Generate Fraudulent Transactions

In [5]:
def generate_fraud_transactions(
    config: SimulationConfig,
    random_generator: np.random.Generator,
) -> pd.DataFrame:
    """Generate crafted fraud patterns seen in UPI risk scenarios."""
    count = config.fraud_transactions
    fraud_base_time = pd.to_datetime(config.start_date) + pd.to_timedelta(config.fraud_start_day, unit="D")
    fraud_timestamps = (
        fraud_base_time
        + pd.to_timedelta(random_generator.integers(0, 365, size=count), unit="D")
        + pd.to_timedelta(
            random_generator.integers(config.fraud_hour_start, config.fraud_hour_end, size=count),
            unit="h",
        )
        + pd.to_timedelta(random_generator.integers(0, 10, size=count), unit="m")
    )

    sender_accounts = make_account_ids(
        prefix="ACC",
        account_numbers=random_generator.integers(1, 5001, size=count),
    )
    receiver_accounts = make_account_ids(
        prefix="RCV",
        account_numbers=np.arange(8001, 8001 + count),
    )

    return pd.DataFrame(
        {
            "transaction_id": make_transaction_ids("FRD", count),
            "timestamp": fraud_timestamps,
            "sender_account_id": sender_accounts,
            "receiver_account_id": receiver_accounts,
            "amount": random_generator.uniform(45000, 200000, size=count).round(2),
            "location_pincode": random_generator.choice(["999999", "888888"], size=count),
            "transaction_type": "P2P",
            "is_fraud": 1,
        }
    )


fraud_transactions = generate_fraud_transactions(config, rng)
fraud_transactions.head()

,transaction_id,timestamp,sender_account_id,receiver_account_id,amount,location_pincode,transaction_type,is_fraud
0,FRD_000001,2024-08-05 03:00:00,ACC_0220,RCV_8001,105335.59,888888,P2P,1
1,FRD_000002,2024-11-01 02:04:00,ACC_4289,RCV_8002,170282.01,888888,P2P,1
2,FRD_000003,2024-10-03 03:04:00,ACC_1667,RCV_8003,45694.25,999999,P2P,1
3,FRD_000004,2024-11-29 03:00:00,ACC_4125,RCV_8004,92254.06,888888,P2P,1
4,FRD_000005,2024-06-09 01:08:00,ACC_1848,RCV_8005,85024.99,999999,P2P,1


## 6. Combine, Shuffle, and Save

In [6]:
def combine_transactions(
    normal_transactions: pd.DataFrame,
    fraud_transactions: pd.DataFrame,
    random_state: int,
) -> pd.DataFrame:
    """Combine normal and fraudulent transactions into one shuffled dataset."""
    transactions = pd.concat(
        [normal_transactions, fraud_transactions],
        ignore_index=True,
    ).sample(frac=1, random_state=random_state)

    transactions = transactions.sort_values("timestamp").reset_index(drop=True)
    transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])
    return transactions


transactions = combine_transactions(
    normal_transactions=normal_transactions,
    fraud_transactions=fraud_transactions,
    random_state=config.random_state,
)

validate_transactions(transactions)

output_path = SYNTHETIC_DATA_DIR / "upi_transactions.csv"
raw_output_path = RAW_DATA_DIR / "upi_transactions.csv"

transactions.to_csv(output_path, index=False)
transactions.to_csv(raw_output_path, index=False)

print(f"Dataset shape: {transactions.shape}")
print(f"Fraud rate: {transactions['is_fraud'].mean():.2%}")
print(f"Saved synthetic data to: {output_path}")
print(f"Saved raw working copy to: {raw_output_path}")

Dataset shape: (10000, 8)
Fraud rate: 3.00%
Saved synthetic data to: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\data\synthetic\upi_transactions.csv
Saved raw working copy to: C:\Users\Prompt\Documents\Real-Time-UPI-Fraud-Detection-System_Notebook\Real-Time-UPI-Fraud-Detection-System\data\raw\upi_transactions.csv


## 7. Quick Data Quality Checks

In [7]:
summary = pd.DataFrame(
    {
        "metric": [
            "total_transactions",
            "normal_transactions",
            "fraud_transactions",
            "unique_senders",
            "unique_receivers",
            "minimum_amount",
            "median_amount",
            "maximum_amount",
        ],
        "value": [
            len(transactions),
            int((transactions["is_fraud"] == 0).sum()),
            int((transactions["is_fraud"] == 1).sum()),
            transactions["sender_account_id"].nunique(),
            transactions["receiver_account_id"].nunique(),
            round(transactions["amount"].min(), 2),
            round(transactions["amount"].median(), 2),
            round(transactions["amount"].max(), 2),
        ],
    }
)

summary

,metric,value
0,total_transactions,10000.00
1,normal_transactions,9700.00
2,fraud_transactions,300.00
3,unique_senders,4341.00
4,unique_receivers,5911.00
5,minimum_amount,17.40
6,median_amount,1650.54
7,maximum_amount,198754.15


In [8]:
transactions.groupby("is_fraud")["amount"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,9700.0,1885.54,1282.06,17.4,932.04,1606.95,2523.80,11010.26
1,300.0,127837.88,45938.57,45581.5,85678.56,125559.63,170070.39,198754.15


In [9]:
transactions.sample(10, random_state=config.random_state)

,transaction_id,timestamp,sender_account_id,receiver_account_id,amount,location_pincode,transaction_type,is_fraud
6252,TXN_009467,2024-02-27 10:10:00,ACC_2575,RCV_1024,2159.37,400001,Merchant,0
4684,TXN_008815,2024-02-12 18:05:00,ACC_1016,RCV_4105,1405.86,700001,P2P,0
1731,TXN_008456,2024-01-16 21:05:00,ACC_3441,RCV_2597,1621.64,700001,P2P,0
4742,TXN_008058,2024-02-13 04:50:00,ACC_0860,RCV_0148,1527.90,560001,P2P,0
4521,TXN_000335,2024-02-11 09:40:00,ACC_3924,RCV_2481,1378.29,110001,Merchant,0
6340,TXN_009545,2024-02-28 07:40:00,ACC_1956,RCV_4134,2186.24,600001,Merchant,0
576,TXN_000619,2024-01-06 12:00:00,ACC_0391,RCV_4386,1995.25,700001,P2P,0
5202,TXN_004813,2024-02-17 09:45:00,ACC_1580,RCV_0561,3771.33,560001,Merchant,0
6363,TXN_000824,2024-02-28 12:15:00,ACC_2926,RCV_2382,2659.91,600001,Merchant,0
439,TXN_009606,2024-01-05 03:30:00,ACC_0816,RCV_3906,533.25,110001,Merchant,0


## 8. Output Files

Generated files:

- `data/synthetic/upi_transactions.csv`
- `data/raw/upi_transactions.csv`

The next notebook can use this CSV for feature engineering, Isolation Forest training, F1-score threshold tuning, and Streamlit deployment.